# 08 TradeBot: Multi-Armed Bandit for Trading Strategy Selection

## 论文参考
- **作者**: Weipeng Zhang
- **年份**: 2021
- **标题**: *TradeBot: Bandit learning for hyper-parameters optimization of high-frequency trading strategy*
- **关键结果**: 平均利润 23,347 RMB
- **摘要**: 本文将交易策略的超参数选择建模为多臂老虎机问题。
  每个"臂"代表一种策略变体（不同参数组合），通过UCB1和Thompson Sampling
  在线学习哪个策略变体在当前市场环境下表现最佳，无需完整的RL状态-动作框架。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 多臂老虎机 (Multi-Armed Bandit, MAB)
给定 $K$ 个臂（策略变体），每轮选择一个臂 $a_t$，获得随机奖励 $r_t \sim \nu_{a_t}$。
目标：最小化累积遗憾 (regret):
$$R_T = T \cdot \mu^* - \sum_{t=1}^T \mu_{a_t}$$

### 1. UCB1 (Upper Confidence Bound)
选择动作:
$$a_t = \arg\max_{k} \left[ \hat{\mu}_k + c \sqrt{\frac{\ln t}{N_k(t)}} \right]$$
- $\hat{\mu}_k$: 臂 $k$ 的经验平均收益
- $N_k(t)$: 臂 $k$ 被选择的次数
- $c$: 探索系数 (通常 $c = \sqrt{2}$)

### 2. Thompson Sampling
对每个臂维护贝叶斯后验分布 $\mu_k \sim \text{Beta}(\alpha_k, \beta_k)$ (伯努利) 或
$\mu_k \sim \mathcal{N}(\hat{\mu}_k, 1/N_k)$ (高斯):
- 采样: $\tilde{\mu}_k \sim P(\mu_k | \text{history})$
- 选择: $a_t = \arg\max_k \tilde{\mu}_k$
- 更新: 观察奖励后更新后验参数

### 策略臂定义
每个臂是一种技术指标交易策略变体:
- Arm 0: MA(5,20) 金叉死叉
- Arm 1: MA(10,30) 金叉死叉
- Arm 2: RSI(14) 超买超卖 (30/70)
- Arm 3: RSI(14) 保守阈值 (40/60)
- Arm 4: Bollinger Band 突破
- Arm 5: MACD 金叉死叉

In [ ]:
# === Cell 3: Bandit 置信区间收缩动画 ===
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

# Simulate 6 arms with different true means
K = 6
true_means = np.array([0.001, 0.003, 0.002, -0.001, 0.0025, 0.0015])
arm_names = ['MA(5,20)', 'MA(10,30)', 'RSI(30/70)', 'RSI(40/60)', 'BollBand', 'MACD']

# Simulate UCB1 running over 500 rounds
T_sim = 500
counts = np.zeros(K)
sum_rewards = np.zeros(K)
frames = []

for t in range(1, T_sim + 1):
    if t <= K:
        arm = t - 1
    else:
        ucb_values = sum_rewards / (counts + 1e-8) + np.sqrt(2 * np.log(t) / (counts + 1e-8))
        arm = np.argmax(ucb_values)

    reward = true_means[arm] + np.random.randn() * 0.01
    counts[arm] += 1
    sum_rewards[arm] += reward

    if t % 20 == 0 or t <= K:
        means = sum_rewards / (counts + 1e-8)
        ci_width = np.sqrt(2 * np.log(t) / (counts + 1e-8))

        frames.append(go.Frame(
            data=[
                go.Bar(x=arm_names, y=means, name='经验均值',
                       marker_color=['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4'],
                       error_y=dict(type='data', array=ci_width, visible=True)),
                go.Scatter(x=arm_names, y=true_means, mode='markers',
                           name='真实均值', marker=dict(color='black', size=12, symbol='star')),
            ],
            name=f'Round {t}'
        ))

fig = go.Figure(data=frames[0].data, frames=frames)
fig.update_layout(
    title=dict(text='UCB1: 各策略臂置信区间随探索次数收缩'),
    xaxis_title='策略臂', yaxis_title='估计收益率',
    height=500, width=900,
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='▶ 播放', method='animate',
             args=[None, {'frame': {'duration': 200}}]),
        dict(label='⏸ 暂停', method='animate',
             args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames],
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.factors import rsi, macd, bollinger_bands, sma
from shared.backtest_engine import Backtester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.constants import *

print('All imports successful.')

In [ ]:
# === Cell 5: 数据获取 ===
stock = get_stock_daily(STOCK_MOUTAI, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

close = stock['close']
high = stock['high']
low = stock['low']

print(f'Stock data: {len(stock)} trading days')
print(f'Date range: {stock.index[0]} ~ {stock.index[-1]}')

In [ ]:
# === Cell 6: 定义策略臂 (6种交易策略变体) ===

def strategy_ma_cross(close, fast, slow):
    """均线金叉死叉策略"""
    ma_fast = sma(close, fast)
    ma_slow = sma(close, slow)
    signals = pd.Series(0, index=close.index)
    signals[ma_fast > ma_slow] = 1
    signals[ma_fast <= ma_slow] = 0
    return signals

def strategy_rsi(close, period=14, lower=30, upper=70):
    """RSI超买超卖策略"""
    rsi_val = rsi(close, period)
    signals = pd.Series(0, index=close.index)
    position = 0
    for i in range(len(rsi_val)):
        if pd.isna(rsi_val.iloc[i]):
            continue
        if rsi_val.iloc[i] < lower and position == 0:
            position = 1
        elif rsi_val.iloc[i] > upper and position == 1:
            position = 0
        signals.iloc[i] = position
    return signals

def strategy_bollinger(close, window=20, num_std=2.0):
    """布林带突破策略"""
    bb = bollinger_bands(close, window, num_std)
    signals = pd.Series(0, index=close.index)
    position = 0
    for i in range(window, len(close)):
        if pd.isna(bb['bb_lower'].iloc[i]):
            continue
        if close.iloc[i] < bb['bb_lower'].iloc[i] and position == 0:
            position = 1
        elif close.iloc[i] > bb['bb_upper'].iloc[i] and position == 1:
            position = 0
        signals.iloc[i] = position
    return signals

def strategy_macd_cross(close):
    """MACD金叉死叉策略"""
    macd_df = macd(close)
    signals = pd.Series(0, index=close.index)
    signals[macd_df['histogram'] > 0] = 1
    signals[macd_df['histogram'] <= 0] = 0
    return signals

# Define all strategy arms
arm_funcs = [
    lambda c: strategy_ma_cross(c, 5, 20),
    lambda c: strategy_ma_cross(c, 10, 30),
    lambda c: strategy_rsi(c, 14, 30, 70),
    lambda c: strategy_rsi(c, 14, 40, 60),
    lambda c: strategy_bollinger(c, 20, 2.0),
    lambda c: strategy_macd_cross(c),
]
arm_names = ['MA(5,20)', 'MA(10,30)', 'RSI(30/70)', 'RSI(40/60)', 'BollBand', 'MACD']
K = len(arm_funcs)

# Pre-compute all strategy signals
all_signals = [func(close) for func in arm_funcs]
print(f'Defined {K} strategy arms: {arm_names}')

In [ ]:
# === Cell 7: UCB1 & Thompson Sampling 在线学习 ===

daily_returns = close.pct_change().fillna(0)
EVAL_WINDOW = 20  # Evaluate arm over 20-day windows
n_windows = len(close) // EVAL_WINDOW

class UCB1:
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c = c
        self.counts = np.zeros(n_arms)
        self.sum_rewards = np.zeros(n_arms)
        self.history = []  # (arm, reward) history

    def select_arm(self, t):
        if t < self.n_arms:
            return t
        means = self.sum_rewards / (self.counts + 1e-8)
        ucb = means + self.c * np.sqrt(np.log(t + 1) / (self.counts + 1e-8))
        return np.argmax(ucb)

    def update(self, arm, reward):
        self.counts[arm] += 1
        self.sum_rewards[arm] += reward
        self.history.append((arm, reward))


class ThompsonSampling:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        # Gaussian Thompson Sampling: N(mu, 1/n)
        self.mu = np.zeros(n_arms)
        self.n = np.zeros(n_arms)
        self.sum_rewards = np.zeros(n_arms)
        self.sum_sq = np.zeros(n_arms)
        self.history = []

    def select_arm(self, t):
        if t < self.n_arms:
            return t
        samples = np.zeros(self.n_arms)
        for k in range(self.n_arms):
            if self.n[k] > 0:
                mean = self.sum_rewards[k] / self.n[k]
                std = 1.0 / np.sqrt(self.n[k] + 1)
                samples[k] = np.random.normal(mean, std)
            else:
                samples[k] = np.random.normal(0, 1)
        return np.argmax(samples)

    def update(self, arm, reward):
        self.n[arm] += 1
        self.sum_rewards[arm] += reward
        self.history.append((arm, reward))


# Run UCB1 and Thompson Sampling
ucb1 = UCB1(K, c=np.sqrt(2))
ts = ThompsonSampling(K)

ucb1_selections = []
ts_selections = []
ucb1_rewards_hist = []
ts_rewards_hist = []
arm_reward_history = {name: [] for name in arm_names}

for w in range(n_windows):
    start_idx = w * EVAL_WINDOW
    end_idx = min(start_idx + EVAL_WINDOW, len(close))
    window_returns = daily_returns.iloc[start_idx:end_idx]

    # Evaluate reward for each arm in this window
    arm_rewards = []
    for k in range(K):
        arm_signal = all_signals[k].iloc[start_idx:end_idx]
        strat_ret = (arm_signal.shift(1).fillna(0) * window_returns).sum()
        arm_rewards.append(strat_ret)

    # UCB1 selects
    ucb1_arm = ucb1.select_arm(w)
    ucb1_reward = arm_rewards[ucb1_arm]
    ucb1.update(ucb1_arm, ucb1_reward)
    ucb1_selections.append(ucb1_arm)
    ucb1_rewards_hist.append(ucb1_reward)

    # Thompson Sampling selects
    ts_arm = ts.select_arm(w)
    ts_reward = arm_rewards[ts_arm]
    ts.update(ts_arm, ts_reward)
    ts_selections.append(ts_arm)
    ts_rewards_hist.append(ts_reward)

print(f'Total windows: {n_windows}')
print(f'\nUCB1 arm selection counts: {dict(zip(arm_names, ucb1.counts.astype(int)))}')
print(f'Thompson arm selection counts: {dict(zip(arm_names, ts.n.astype(int)))}')

In [ ]:
# === Cell 8: 构建Bandit复合信号并回测 ===

def build_bandit_signal(selections, all_signals, eval_window, total_len):
    """根据bandit每期选择拼接信号"""
    combined = pd.Series(0, index=close.index[:total_len])
    for w, arm in enumerate(selections):
        start = w * eval_window
        end = min(start + eval_window, total_len)
        combined.iloc[start:end] = all_signals[arm].iloc[start:end]
    return combined

total_len = len(close)
ucb1_signal = build_bandit_signal(ucb1_selections, all_signals, EVAL_WINDOW, total_len)
ts_signal = build_bandit_signal(ts_selections, all_signals, EVAL_WINDOW, total_len)

bench_close = benchmark['close'] if 'close' in benchmark.columns else None
bt = Backtester(initial_capital=INITIAL_CAPITAL, t_plus_1=True)

# Backtest UCB1
result_ucb = bt.run(close, ucb1_signal, bench_close)
# Backtest Thompson
result_ts = bt.run(close, ts_signal, bench_close)

# Backtest individual arms for comparison
individual_results = {}
for k, name in enumerate(arm_names):
    res = bt.run(close, all_signals[k], bench_close)
    individual_results[name] = res

print('=== UCB1 Bandit ===')
for k, v in result_ucb['metrics'].items():
    print(f'  {k}: {v}')

print('\n=== Thompson Sampling ===')
for k, v in result_ts['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# === Cell 9: 可视化 ===

# 1. All strategies cumulative comparison
all_returns = {'UCB1': result_ucb['returns'], 'Thompson': result_ts['returns']}
for name in arm_names:
    all_returns[name] = individual_results[name]['returns']
plot_cumulative_comparison(all_returns, title='Bandit vs 个体策略 - 累计收益对比')

# 2. Arm selection frequency over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# UCB1 selections
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
window_range = range(len(ucb1_selections))
axes[0].scatter(window_range, ucb1_selections, c=[colors[s] for s in ucb1_selections],
                s=30, alpha=0.7)
axes[0].set_yticks(range(K))
axes[0].set_yticklabels(arm_names)
axes[0].set_title('UCB1 - 策略臂选择序列', fontsize=13, fontweight='bold')
axes[0].set_xlabel('时间窗口')
axes[0].grid(True, alpha=0.3)

# Thompson selections
axes[1].scatter(window_range, ts_selections, c=[colors[s] for s in ts_selections],
                s=30, alpha=0.7)
axes[1].set_yticks(range(K))
axes[1].set_yticklabels(arm_names)
axes[1].set_title('Thompson Sampling - 策略臂选择序列', fontsize=13, fontweight='bold')
axes[1].set_xlabel('时间窗口')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Arm selection pie charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(ucb1.counts, labels=arm_names, colors=colors, autopct='%1.0f%%',
            startangle=90)
axes[0].set_title('UCB1 选择分布', fontsize=13, fontweight='bold')
axes[1].pie(ts.n, labels=arm_names, colors=colors, autopct='%1.0f%%',
            startangle=90)
axes[1].set_title('Thompson Sampling 选择分布', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 4. Cumulative reward comparison
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(np.cumsum(ucb1_rewards_hist), 'b-', linewidth=2, label='UCB1 累计收益')
ax.plot(np.cumsum(ts_rewards_hist), 'r-', linewidth=2, label='Thompson 累计收益')
ax.set_title('Bandit 累计收益对比', fontsize=14, fontweight='bold')
ax.set_xlabel('时间窗口')
ax.set_ylabel('累计收益率')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 5. Metrics tables
plot_equity_curve(result_ucb['equity_curve'], title='UCB1 Bandit - 权益曲线')
plot_metrics_table(result_ucb['metrics'], title='UCB1 Bandit - 绩效指标')
plot_metrics_table(result_ts['metrics'], title='Thompson Sampling - 绩效指标')

## 结果讨论

### 核心发现
1. **Bandit简洁有效**: 相比完整RL，多臂老虎机不需要状态表示和策略网络，计算成本极低
2. **自适应选择**: UCB1和Thompson Sampling都能逐渐收敛到最优策略臂
3. **探索-利用权衡**: UCB1通过确定性上界探索，Thompson通过随机采样探索

### UCB1 vs Thompson Sampling
- UCB1: 确定性策略，上界收紧速度均匀，适合稳态环境
- Thompson: 概率性策略，初期探索更积极，在非平稳环境中更灵活

### 与论文对比
- Zhang (2021) 使用bandit优化高频策略的具体超参数（阈值、窗口等）
- 本实现将不同策略类型作为臂，更易理解核心概念
- 论文报告平均利润23,347 RMB，核心优势在于零样本切换成本

### 改进方向
- 使用 Contextual Bandit: 根据市场状态选择臂
- 非平稳 Bandit (Sliding Window UCB / Discounted UCB)
- 将超参数连续化为 Bayesian Optimization